# Real Drive-Time Isochrones using OSRM (Open Source)

Uses OSRM (Open Source Routing Machine) public API for real road network routing.
- **100% Free & Open Source**
- **No installation needed** - uses public OSRM demo server
- **Real routing** using actual road network
- **No API key required**

Note: The public server has rate limits. For high-volume production use, consider self-hosting OSRM.

## Parameters

In [ ]:
dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("bronze_schema", "geo_bronze")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")
dbutils.widgets.text("lce_locations_table", "lce_locations_mass")
dbutils.widgets.dropdown("input_source", "lce", ["lce", "convenience"], "Input Source")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
lce_locations_table = dbutils.widgets.get("lce_locations_table")
input_source = dbutils.widgets.get("input_source")

# Determine input and output tables based on source
if input_source == "lce":
    input_table = f"{catalog}.{bronze_schema}.{lce_locations_table}"
    output_table = f"{catalog}.{silver_schema}.isochrones_lce"
    store_type_label = "Little Caesars"
elif input_source == "convenience":
    input_table = f"{catalog}.{silver_schema}.pois_convenience"
    output_table = f"{catalog}.{silver_schema}.isochrones_convenience"
    store_type_label = "Convenience Store"
else:
    raise ValueError(f"Unknown input_source: {input_source}")

# Use public OSRM demo server
OSRM_BASE_URL = "https://router.project-osrm.org"

print(f"Input source: {input_source}")
print(f"Input table: {input_table}")
print(f"Output table: {output_table}")

## Setup - Test Connection to Public OSRM API

In [ ]:
import requests
import time

# Test connection to public OSRM server
print(f"Testing connection to {OSRM_BASE_URL}...")

try:
    # Test with a simple route in Boston area
    test_url = f"{OSRM_BASE_URL}/route/v1/driving/-71.0589,42.3601;-71.0603,42.3584"
    response = requests.get(test_url, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        if 'routes' in data:
            print("✓ Successfully connected to public OSRM server")
            print(f"  Server: {OSRM_BASE_URL}")
            print(f"  Test route duration: {data['routes'][0]['duration']:.1f} seconds")
        else:
            print("⚠ Connected but unexpected response format")
    else:
        print(f"⚠ Server returned status {response.status_code}")
        raise Exception(f"OSRM server not available: {response.status_code}")
        
except Exception as e:
    print(f"❌ Could not connect to OSRM server: {e}")
    print("\nNote: The public OSRM demo server may have rate limits.")
    print("If this fails, consider using Mapbox API or self-hosting OSRM.")
    raise

# Add a small delay to be respectful of the public server
time.sleep(1)

## Load Locations and Urbanicity

In [ ]:
from pyspark.sql.functions import col, expr, broadcast, lit

# Read locations from the appropriate table
locations = spark.table(input_table)

# Auto-detect columns (handles both LCE and POI schemas)
columns = locations.columns
id_col = next((c for c in columns if c in ['store_number', 'point_id', 'id', 'location_id', 'poi_id']), columns[0])
lat_col = next((c for c in columns if c in ['latitude', 'lat', 'y']), None)
lon_col = next((c for c in columns if c in ['longitude', 'lon', 'lng', 'x']), None)
city_col = next((c for c in columns if c in ['city', 'municipality']), None)
state_col = next((c for c in columns if c in ['state', 'region', 'state_abbr']), None)
name_col = next((c for c in columns if c in ['name', 'store_name']), None)

if not lat_col or not lon_col:
    raise ValueError(f"Cannot find lat/lon columns. Available: {columns}")

# Standardize columns - include city and state
locations_std = locations.select(
    col(id_col).alias("location_id"),
    col(lat_col).alias("latitude"),
    col(lon_col).alias("longitude"),
    (col(name_col) if name_col else lit(store_type_label)).alias("store_type"),
    (col(city_col) if city_col else lit(None)).alias("city"),
    (col(state_col) if state_col else lit(None)).alias("state")
).filter(col("latitude").isNotNull() & col("longitude").isNotNull())

# Simple fixed 5-minute drive time for all locations
locations_with_times = locations_std.withColumn(
    "urbanicity_category", lit("standard")
).withColumn(
    "drive_time_minutes", lit(5)  # Fixed 5-minute drive time
)

print(f"Source: {input_source}")
print(f"Loaded {locations_with_times.count()} locations from {input_table}")
print(f"Using fixed 5-minute drive time")
display(locations_with_times.limit(5))

## Generate Isochrones using OSRM

In [ ]:
import requests
import json
from shapely.geometry import LineString, Polygon
from shapely.ops import unary_union
import math
import time

def get_osrm_isochrone(lon, lat, minutes, base_url=OSRM_BASE_URL):
    """
    Generate isochrone using OSRM API

    OSRM doesn't have native isochrone support, so we:
    1. Generate routes in multiple directions
    2. Find points at target drive time
    3. Create polygon from those points
    """

    # Generate points in a circle around location
    num_directions = 32  # More directions = smoother polygon
    distance_km = minutes * 1.0  # Rough estimate: 60 km/h average speed

    points_at_time = []

    for i in range(num_directions):
        angle = (2 * math.pi * i) / num_directions

        # Calculate destination point (rough approximation)
        lat_offset = (distance_km / 111.32) * math.cos(angle)
        lon_offset = (distance_km / (111.32 * math.cos(math.radians(lat)))) * math.sin(angle)

        dest_lon = lon + lon_offset
        dest_lat = lat + lat_offset

        # Get route from OSRM
        try:
            url = f"{base_url}/route/v1/driving/{lon},{lat};{dest_lon},{dest_lat}"
            params = {
                'overview': 'full',
                'geometries': 'geojson'
            }

            response = requests.get(url, params=params, timeout=10)

            if response.status_code == 200:
                data = response.json()

                if 'routes' in data and len(data['routes']) > 0:
                    route = data['routes'][0]
                    route_duration = route['duration'] / 60  # Convert to minutes
                    coordinates = route['geometry']['coordinates']

                    # Find point closest to target time
                    if route_duration > 0:
                        target_ratio = minutes / route_duration
                        if target_ratio <= 1:
                            target_idx = int(len(coordinates) * target_ratio)
                            target_idx = min(target_idx, len(coordinates) - 1)
                            points_at_time.append(coordinates[target_idx])
            
            # Small delay to respect rate limits on public server
            time.sleep(0.05)
            
        except Exception as e:
            # Silently continue on errors (some routes may fail)
            continue

    # Create polygon from points
    if len(points_at_time) >= 3:
        try:
            polygon = Polygon(points_at_time)
            polygon = polygon.convex_hull

            # Convert to WKT
            coords_str = ', '.join([f"{lon} {lat}" for lon, lat in polygon.exterior.coords])
            return f"POLYGON (({coords_str}))"
        except:
            return None

    return None

In [ ]:
# Test single isochrone
test_location = locations_with_times.first()
print(f"Testing isochrone generation for: {test_location.location_id}")
print(f"Location: ({test_location.latitude}, {test_location.longitude})")
print(f"Drive time: {test_location.drive_time_minutes} minutes")
print(f"\nGenerating isochrone with {32} route samples...")

wkt = get_osrm_isochrone(test_location.longitude, test_location.latitude, test_location.drive_time_minutes)

if wkt:
    print("\n✓ Test isochrone generated successfully!")
    print(f"  WKT polygon length: {len(wkt)} characters")
    print(f"  Using public OSRM server: {OSRM_BASE_URL}")
else:
    print("\n⚠ Test failed - isochrone could not be generated")
    print("  This may be due to rate limits or connectivity issues")

## Generate All Isochrones

In [ ]:
from pyspark.sql import Row
import time

location_rows = locations_with_times.collect()
print(f"Generating isochrones for {len(location_rows)} locations...")
print(f"Using public OSRM server: {OSRM_BASE_URL}")
print(f"Estimated time: ~{len(location_rows) * 2} seconds (with rate limiting)")
print("")

results = []
start_time = time.time()

for i, row in enumerate(location_rows):
    if i % 5 == 0:
        elapsed = time.time() - start_time
        if i > 0:
            avg_time = elapsed / i
            remaining = (len(location_rows) - i) * avg_time
            print(f"Progress: {i}/{len(location_rows)} ({i/len(location_rows)*100:.1f}%) - "
                  f"Elapsed: {elapsed:.1f}s - ETA: {remaining:.1f}s")

    wkt = get_osrm_isochrone(row.longitude, row.latitude, row.drive_time_minutes)

    if wkt:
        results.append(Row(
            location_id=row.location_id,
            latitude=row.latitude,
            longitude=row.longitude,
            store_type=row.store_type,
            city=row.city,
            state=row.state,
            urbanicity_category=row.urbanicity_category,
            drive_time_minutes=row.drive_time_minutes,
            geometry_wkt=wkt
        ))

total_time = time.time() - start_time
print(f"\n✅ Generated {len(results)}/{len(location_rows)} isochrones successfully")
print(f"   Total time: {total_time:.1f} seconds")
print(f"   Average: {total_time/len(location_rows):.2f} seconds per location")

## Save to Delta

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import current_timestamp

# Create DataFrame
isochrone_schema = StructType([
    StructField("location_id", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("store_type", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("urbanicity_category", StringType(), True),
    StructField("drive_time_minutes", IntegerType(), False),
    StructField("geometry_wkt", StringType(), False)
])

isochrones_df = spark.createDataFrame(results, schema=isochrone_schema)

# Convert to geometry and add metadata
isochrones_final = (
    isochrones_df
    .withColumn("geometry", expr("ST_GeomFromText(geometry_wkt, 4326)"))
    .withColumn("area_sqkm", expr("ST_Area(geometry) / 1000000"))
    .withColumn("created_timestamp", current_timestamp())
    .withColumn("routing_provider", lit("osrm"))
    .drop("geometry_wkt")
)

# Save to silver schema (output_table already contains full path)
(
    isochrones_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"Saved {len(results)} isochrones to {output_table}")

## Summary

In [ ]:
display(spark.sql(f"""
    SELECT
        urbanicity_category,
        drive_time_minutes,
        COUNT(*) as count,
        ROUND(AVG(area_sqkm), 2) as avg_area_sqkm
    FROM {output_table}
    GROUP BY urbanicity_category, drive_time_minutes
    ORDER BY urbanicity_category
"""))

## Visualize Isochrones with Folium


In [ ]:
import folium
from shapely import wkt as shapely_wkt
import json

# Create map centered on Massachusetts
ma_center = [42.4072, -71.3824]  # Massachusetts center
m = folium.Map(location=ma_center, zoom_start=8, tiles='OpenStreetMap')

# Add isochrone polygons
print(f"Adding {len(results)} isochrone polygons to map...")
for i, result in enumerate(results):
    # Parse WKT to get coordinates
    polygon = shapely_wkt.loads(result.geometry_wkt)
    coords = [[lat, lon] for lon, lat in polygon.exterior.coords]
    
    # Add polygon to map
    folium.Polygon(
        locations=coords,
        color='#FF6B35',  # Orange border
        fillColor='#FF6B35',
        fillOpacity=0.2,
        weight=2,
        popup=f"Store: {result.location_id}<br>{result.drive_time_minutes} min drive time"
    ).add_to(m)

# Add store locations as markers
print(f"Adding {len(location_rows)} store locations to map...")
for location in location_rows:
    folium.CircleMarker(
        location=[location.latitude, location.longitude],
        radius=6,
        popup=f"<b>Little Caesars</b><br>Store: {location.location_id}<br>{location.drive_time_minutes} min isochrone",
        color='#C1121F',  # Red border
        fillColor='#C1121F',
        fillOpacity=0.8,
        weight=2
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 180px; height: 120px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p style="margin-bottom: 5px;"><b>Legend</b></p>
<p style="margin: 5px 0;"><span style="color: #C1121F;">●</span> Little Caesars Store</p>
<p style="margin: 5px 0;"><span style="background-color: rgba(255,107,53,0.3); padding: 0 8px;">█</span> 5-min Drive Time</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f"\n✅ Map created with {len(results)} isochrones and {len(location_rows)} stores")
print(f"   Coverage area: Massachusetts")
print(f"   Drive time: {results[0].drive_time_minutes} minutes")

# Display the map
m


## Summary

In [ ]:
display(spark.sql(f"""
    SELECT
        urbanicity_category,
        drive_time_minutes,
        COUNT(*) as count,
        ROUND(AVG(area_sqkm), 2) as avg_area_sqkm
    FROM {output_table}
    GROUP BY urbanicity_category, drive_time_minutes
    ORDER BY urbanicity_category
"""))